In [1]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

import pandas as pd
import groq

In [2]:
df_items=pd.read_json("./../data/meta_CDs_and_Vinyl_2022_2023_with_category_ratings_100_sample_1000.jsonl", lines=True)

In [3]:
df_items.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
0,Digital Music,Queen II[LP],4.7,1835,[],[Queen II is the second studio album from the ...,29.33,[{'thumb': 'https://m.media-amazon.com/images/...,[],Queen Format: Vinyl,"[CDs & Vinyl, Rock]","{'Language': 'English', 'Product Dimensions': ...",B0BDFW4VRB,NaN
1,Digital Music,The Elephants of Mars Limited Orange,4.7,125,[],"[It moves, it swings, it rocks!Satriani and hi...",26.98,[{'thumb': 'https://m.media-amazon.com/images/...,[],Joe Satriani Format: Vinyl,"[CDs & Vinyl, Rock]",{'Product Dimensions': '12 x 12.3 x 0.4 inches...,B09Q98326Z,NaN
2,Digital Music,The Love Invention Heavyweight Black,4.5,158,[],[Alison Goldfrapp has set a towering bar for B...,20.49,[{'thumb': 'https://m.media-amazon.com/images/...,[],Alison Goldfrapp (Artist) Format: Vinyl,"[CDs & Vinyl, Dance & Electronic, Electronica]","{'Language': 'English', 'Product Dimensions': ...",B0BYHFG53W,NaN
3,Digital Music,Now That's What I Call A Love Song / Various,4.5,120,[],[NOW Music is proud to present 80 of the great...,25.68,[{'thumb': 'https://m.media-amazon.com/images/...,[],Various Artists (Artist) Format: Audio CD,"[CDs & Vinyl, Rock]",{'Package Dimensions': '5.51 x 4.92 x 0.51 inc...,B0BS1SHT91,NaN
4,Digital Music,SZNZ: Summer,4.4,120,[],"[SZNZ: Summer, the second part in Weezer's fou...",8.79,[{'thumb': 'https://m.media-amazon.com/images/...,[],Weezer Format: Audio CD,"[CDs & Vinyl, Indie & Alternative, Indie & Lo-Fi]","{'Language': 'English', 'Product Dimensions': ...",B0B2ZRV4FD,NaN


In [4]:
#swap features and description coloumns
df_items["features"], df_items["description"] = df_items["description"], df_items["features"]

In [5]:
list(df_items["features"].items())[0]

(0,
 ['Queen II is the second studio album from the iconic British rock band. Originally released in 1974, it reached #5 on the U.K. albums chart. This reissue is pressed on 180-gram vinyl and sourced from the original master tapes.'])

In [6]:
df_items.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
0,Digital Music,Queen II[LP],4.7,1835,[Queen II is the second studio album from the ...,[],29.33,[{'thumb': 'https://m.media-amazon.com/images/...,[],Queen Format: Vinyl,"[CDs & Vinyl, Rock]","{'Language': 'English', 'Product Dimensions': ...",B0BDFW4VRB,NaN
1,Digital Music,The Elephants of Mars Limited Orange,4.7,125,"[It moves, it swings, it rocks!Satriani and hi...",[],26.98,[{'thumb': 'https://m.media-amazon.com/images/...,[],Joe Satriani Format: Vinyl,"[CDs & Vinyl, Rock]",{'Product Dimensions': '12 x 12.3 x 0.4 inches...,B09Q98326Z,NaN
2,Digital Music,The Love Invention Heavyweight Black,4.5,158,[Alison Goldfrapp has set a towering bar for B...,[],20.49,[{'thumb': 'https://m.media-amazon.com/images/...,[],Alison Goldfrapp (Artist) Format: Vinyl,"[CDs & Vinyl, Dance & Electronic, Electronica]","{'Language': 'English', 'Product Dimensions': ...",B0BYHFG53W,NaN
3,Digital Music,Now That's What I Call A Love Song / Various,4.5,120,[NOW Music is proud to present 80 of the great...,[],25.68,[{'thumb': 'https://m.media-amazon.com/images/...,[],Various Artists (Artist) Format: Audio CD,"[CDs & Vinyl, Rock]",{'Package Dimensions': '5.51 x 4.92 x 0.51 inc...,B0BS1SHT91,NaN
4,Digital Music,SZNZ: Summer,4.4,120,"[SZNZ: Summer, the second part in Weezer's fou...",[],8.79,[{'thumb': 'https://m.media-amazon.com/images/...,[],Weezer Format: Audio CD,"[CDs & Vinyl, Indie & Alternative, Indie & Lo-Fi]","{'Language': 'English', 'Product Dimensions': ...",B0B2ZRV4FD,NaN


In [7]:
list(df_items["images"].items())[0]

(0,
 [{'thumb': 'https://m.media-amazon.com/images/I/41wmg-DUsDL._SS40_.jpg',
   'large': 'https://m.media-amazon.com/images/I/41wmg-DUsDL.jpg',
   'variant': 'MAIN',
   'hi_res': 'https://m.media-amazon.com/images/I/61VO5mwSWcL._SL1500_.jpg'}])

### Preprocess Title and features

In [8]:
def preprocess_features(row):
    return f"{row['title']} {' '.join(row['features'])}"

In [9]:
def extract_first_large_image(row):
    return row["images"][0].get("large", "")

In [10]:
df_items["description"] = df_items.apply(preprocess_features, axis=1)
df_items["image"] = df_items.apply(extract_first_large_image, axis=1)

In [11]:
df_items.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,image
0,Digital Music,Queen II[LP],4.7,1835,[Queen II is the second studio album from the ...,Queen II[LP] Queen II is the second studio alb...,29.33,[{'thumb': 'https://m.media-amazon.com/images/...,[],Queen Format: Vinyl,"[CDs & Vinyl, Rock]","{'Language': 'English', 'Product Dimensions': ...",B0BDFW4VRB,NaN,https://m.media-amazon.com/images/I/41wmg-DUsD...
1,Digital Music,The Elephants of Mars Limited Orange,4.7,125,"[It moves, it swings, it rocks!Satriani and hi...","The Elephants of Mars Limited Orange It moves,...",26.98,[{'thumb': 'https://m.media-amazon.com/images/...,[],Joe Satriani Format: Vinyl,"[CDs & Vinyl, Rock]",{'Product Dimensions': '12 x 12.3 x 0.4 inches...,B09Q98326Z,NaN,https://m.media-amazon.com/images/I/31uqhdHGwB...
2,Digital Music,The Love Invention Heavyweight Black,4.5,158,[Alison Goldfrapp has set a towering bar for B...,The Love Invention Heavyweight Black Alison Go...,20.49,[{'thumb': 'https://m.media-amazon.com/images/...,[],Alison Goldfrapp (Artist) Format: Vinyl,"[CDs & Vinyl, Dance & Electronic, Electronica]","{'Language': 'English', 'Product Dimensions': ...",B0BYHFG53W,NaN,https://m.media-amazon.com/images/I/41Y1Uqko5C...
3,Digital Music,Now That's What I Call A Love Song / Various,4.5,120,[NOW Music is proud to present 80 of the great...,Now That's What I Call A Love Song / Various N...,25.68,[{'thumb': 'https://m.media-amazon.com/images/...,[],Various Artists (Artist) Format: Audio CD,"[CDs & Vinyl, Rock]",{'Package Dimensions': '5.51 x 4.92 x 0.51 inc...,B0BS1SHT91,NaN,https://m.media-amazon.com/images/I/51QqhSYl+D...
4,Digital Music,SZNZ: Summer,4.4,120,"[SZNZ: Summer, the second part in Weezer's fou...","SZNZ: Summer SZNZ: Summer, the second part in ...",8.79,[{'thumb': 'https://m.media-amazon.com/images/...,[],Weezer Format: Audio CD,"[CDs & Vinyl, Indie & Alternative, Indie & Lo-Fi]","{'Language': 'English', 'Product Dimensions': ...",B0B2ZRV4FD,NaN,https://m.media-amazon.com/images/I/51Fv6K5rZ8...


#### Sample 50 items from dataset (for quick devlopment)

In [12]:
df_sample=df_items.sample(50, random_state=42)

In [13]:
len(df_sample)

50

In [14]:
data_to_embed=df_sample[["description", "image","rating_number","price","average_rating","parent_asin"]].to_dict(orient="records")

In [15]:
data_to_embed 

[{'description': "I Am The Moon: IV. Farewell The fifth studio project from the Tedeschi Trucks band is the most ambitious and, at the same time, intimate record that America's best rock n' roll big band has ever made: a genuinely epic undertaking in four episodes and 24 songs inspired by classical literature but emotionally driven by the immediate drama, isolation and mourning of the pandemic era. CD Softpak.",
  'image': 'https://m.media-amazon.com/images/I/315JEh2Xh9L.jpg',
  'rating_number': 309,
  'price': 10.98,
  'average_rating': 4.7,
  'parent_asin': 'B09XVCWQGL'},
 {'description': 'When Christmas Comes Around... GRAMMY-winning global superstarKellyClarksonhas announcedWhen Christmas Comes Around…on vinyl, her ninth studio album arrived viaAtlanticRecordsin 2021. Led by the fabulously bold lead single “Christmas Isn’t Canceled (Just You),” the album explores a wide range of holiday emotions and experiences anchored by Clarkson’s incomparable vocal prowess. From the show-stoppi

### define the embedding function

In [16]:
### define the embedding function minilm
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')
def embed_text(text):
    return model.encode(text).tolist()
# Example usage
text = "This is an example sentence."
embedding = embed_text(text)
print(embedding)




c:\Users\Loq\Documents\CRAP\end to end aibootcamp\code\handsON\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4213.16it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[0.09812459349632263, 0.06781266629695892, 0.06252316385507584, 0.0950847640633583, 0.03664756938815117, -0.003984653856605291, 0.0074775186367332935, -0.013231487944722176, 0.06288367509841919, 0.022495532408356667, 0.07269581407308578, -0.03127426281571388, 0.04635513573884964, -0.012554490007460117, 0.047814738005399704, -0.004910328891128302, 0.04941995441913605, -0.06410927325487137, -0.09696577489376068, 0.03288881480693817, 0.054104436188936234, 0.03532859683036804, 0.0330505445599556, 0.014699398539960384, -0.03343060612678528, -0.025615902617573738, -0.05079213157296181, 0.07325458526611328, 0.11027402430772781, -0.02966185286641121, -0.06755709648132324, -0.03057153709232807, 0.039560288190841675, 0.04547598958015442, 0.015996240079402924, 0.03855039179325104, -0.010954095050692558, 0.08483570069074631, -0.04428712651133537, -0.006796381436288357, 0.009425639174878597, 5.066889207228087e-05, 0.0013035230804234743, -0.011969797313213348, 0.013645161874592304, -0.08417426794767

In [17]:
len(embedding)

384

In [18]:
def get_embedding(text,model_name="all-MiniLM-L6-v2"):
    model = SentenceTransformer(model_name)
    return model.encode(text).tolist()

### create qdrant collections


In [20]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [21]:
qdrant_client.create_collection(
    collection_name="amazon-items-collection-00",
    vectors_config=VectorParams(size=384, distance=Distance.COSINE)
)

True

### Embed Data

#### Test


In [22]:
pointstruct=PointStruct(
    id=0,
    vector=get_embedding("Test Text"),
    payload={
        "text": "Test Text",
        "model": "all-MiniLM-L6-v2"
    },
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7955.17it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [23]:
pointstruct

PointStruct(id=0, vector=[0.0357932411134243, 0.08075187355279922, -0.022894641384482384, 0.03583967685699463, -0.01652579754590988, 0.02314174920320511, 0.06864471733570099, 0.013322337530553341, 0.018502315506339073, -0.012461098842322826, 0.07817526906728745, -0.012404758483171463, 0.03556797653436661, 0.007992545142769814, 0.03585612773895264, -0.0376138761639595, 0.04076767340302467, -0.017905496060848236, -0.03391685336828232, -0.05569085106253624, 0.02311631664633751, 0.04743529111146927, 0.017255960032343864, 0.04447336494922638, -0.01573190838098526, 0.10072934627532959, -0.08943486213684082, 0.02642359770834446, 0.026291079819202423, -0.021334869787096977, -0.021611902862787247, 0.03308769315481186, 0.07231839746236801, 0.07236592471599579, 0.057040996849536896, -0.003308030543848872, -0.03990710526704788, 0.03111330047249794, 0.017501384019851685, 0.03648112714290619, -0.06317993253469467, -0.11546606570482254, 0.0389338918030262, 0.02778879553079605, 0.09715817868709564, -0

### Amazon data

In [24]:
pointstructs=[]

for i,data in enumerate(data_to_embed):
    embedding=get_embedding(data["description"])
    pointstructs.append(
        PointStruct(
            id=i,
            vector=embedding,
            payload=data
        )
    )

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7194.11it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8264.40it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5894.65it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embedd

In [26]:
len(pointstructs)

50

### write embedded data to Qdrant

In [27]:
qdrant_client.upsert(
    collection_name="amazon-items-collection-00",
    wait=True,
    points=pointstructs
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

### def function for data retreval

In [30]:
def retrieve_data(query, top_k=5):
    query_embedding=get_embedding(query)
    search_result=qdrant_client.query_points(
        collection_name="amazon-items-collection-00",
        query=query_embedding,
        limit=top_k,
    )
    return search_result

### test retreive

In [31]:
## test retreive

retrieve_data("Which album has great sound quality and a unique blend of genres?", top_k=10).points

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7870.53it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[ScoredPoint(id=11, version=1, score=0.40321946, payload={'description': "MERCY . What does John Cale have that the rest of us don’t—some gene that engenders infinite restlessness, a rapacious mind that is never satisfied? For nearly 60 years, or at least since he was a young Welshman who moved to New York and joined The Velvet Underground, Cale has been reinventing his music with dazzling and inspiring regularity. Once again, here is Cale, reimagining how his music is made, sounds, and even works. His engrossing 12-track MERCY—his first full album of new tunes in a decade—moves through true dark-night-of-the-soul electronic blues toward vulnerable love songs and hopeful considerations for the future with the help of some of music’s most curious young minds. 1. MERCY feat. Laurel Halo2. MARILYN MONROE'S LEGS (beauty elsewhere) feat. Actress3. NOISE OF YOU4. STORY OF BLOOD feat. Weyes Blood5. TIME STANDS STILL feat. Sylvan Esso6. MOONSTRUCK (Nico's Song)7. EVERLASTING DAYS feat. Animal 